In [1]:
import os
import re
import numpy as np
import pandas as pd
 
# 1. Load the data
df_base = pd.read_csv("Baseline_radiomics_values_LCIO.csv", low_memory=False)
df_fu = pd.read_csv("fu1_radiomic_values_LCIO.csv", low_memory=False)

In [2]:
# 2. Mask classification 
def prep_dataframe(df):
    df = df.copy()
    id_col = "ID" if "ID" in df.columns else "PatientID"
 
    df["Patient_Num"] = df[id_col].str.extract(r"(\d+)").astype(float)
    df["mask_base"] = df["Mask"].apply(
        lambda x: re.sub(r"\.nii\.gz$", "", str(os.path.basename(str(x)))).strip()
    )
 
    conditions = [
        df["mask_base"].str.contains(r'(?i)_?[Pp]$|pre_Mp|post_Mp|Mp', regex=True),
        df["mask_base"].str.contains(r'(?i)_?[Ss]$|pre_Ms|post_Ms|Ms', regex=True),
        df["mask_base"].str.contains(r'(?i)seg|mc_seg|_mc$|_M$|main', regex=True)
    ]
    choices = ["Mp", "Ms", "main_seg"]
 
    df["Raw_Mask_Type"] = np.select(conditions, choices, default="plain")
    df["Mask_Type"] = np.where(df["Raw_Mask_Type"].isin(["plain", "main_seg"]), "main", df["Raw_Mask_Type"])
    df["Priority"] = np.where(df["Raw_Mask_Type"] == "main_seg", 1, 2)
 
    df = df.sort_values(["Patient_Num", "Priority"]).drop_duplicates(
        subset=["Patient_Num", "Mask_Type"], keep="first"
    ).reset_index(drop=True)
 
    return df
 
df_base = prep_dataframe(df_base)
df_fu = prep_dataframe(df_fu)
 
print("BASELINE Mask_Type counts:\n", df_base["Mask_Type"].value_counts())
print("\nFOLLOW-UP Mask_Type counts:\n", df_fu["Mask_Type"].value_counts())

BASELINE Mask_Type counts:
 Mask_Type
main    158
Ms      158
Mp      158
Name: count, dtype: int64

FOLLOW-UP Mask_Type counts:
 Mask_Type
main    130
Ms      130
Mp      130
Name: count, dtype: int64


In [3]:
NON_FEATURE_PREFIXES = ("ID", "Image", "Mask", "diagnostics")
 
def split_diag_feat(df):
    feat_cols = [c for c in df.columns if not c.startswith(NON_FEATURE_PREFIXES)]
    diag_cols = [c for c in df.columns if c.startswith(NON_FEATURE_PREFIXES) and c not in ("Mask",)]
   
    diag_cols = [c for c in diag_cols if c != "Mask"]
    return df[diag_cols].copy(), df[feat_cols].apply(pd.to_numeric, errors="coerce")
 
def make_delta_separated(mask_label):
    pre = df_base[df_base["Mask_Type"] == mask_label].sort_values("Patient_Num")
    post = df_fu[df_fu["Mask_Type"] == mask_label].sort_values("Patient_Num")
    
    common_pts = np.intersect1d(pre["Patient_Num"], post["Patient_Num"])
    pre = pre[pre["Patient_Num"].isin(common_pts)].set_index("Patient_Num")
    post = post[post["Patient_Num"].isin(common_pts)].set_index("Patient_Num")
    
    # Identify feature columns based on PyRadiomics naming (this accounts for every feature in pyradiomics config)
    feature_prefixes = ('original_', 'log-sigma-', 'wavelet-', 'square_', 'squareroot_',
                     'logarithm_', 'exponential_', 'gradient_', 'lbp-')
    
    pre_features = pre[[c for c in pre.columns if c.startswith(feature_prefixes)]].apply(pd.to_numeric, errors="coerce")
    post_features = post[[c for c in post.columns if c.startswith(feature_prefixes)]].apply(pd.to_numeric, errors="coerce")
    
    # Align columns to ensure the caculation is performed on matching feature names
    pre_features, post_features = pre_features.align(post_features, join='inner', axis=1)
    
    # Epsilon
    epsilon = (0.01 * pre_features.abs().median()).replace(0, 1e-8)
    
    # 1. ABSOLUTE
    abs_vals = post_features - pre_features
    df_abs = abs_vals.add_prefix(f"{mask_label}_abs_").reset_index()
    
    # 2. RELATIVE
    rel_vals = (post_features - pre_features) / (pre_features.abs() + epsilon)
    df_rel = rel_vals.add_prefix(f"{mask_label}_rel_").reset_index()
    
    # 3. SYMMETRIC
    sym_vals = (2 * (post_features - pre_features)) / (post_features.abs() + pre_features.abs() + epsilon)
    df_sym = sym_vals.add_prefix(f"{mask_label}_sym_").reset_index()
    
    return df_abs, df_rel, df_sym

In [4]:
# 4. Compute deltas
main_abs, main_rel, main_sym = make_delta_separated("main")
Mp_abs, Mp_rel, Mp_sym = make_delta_separated("Mp")
Ms_abs, Ms_rel, Ms_sym = make_delta_separated("Ms")
 
# 5. Merge
data_abs = main_abs.merge(Mp_abs, on="Patient_Num", how="outer") \
                   .merge(Ms_abs, on="Patient_Num", how="outer")
 
data_rel = main_rel.merge(Mp_rel, on="Patient_Num", how="outer") \
                   .merge(Ms_rel, on="Patient_Num", how="outer")
 
data_sym = main_sym.merge(Mp_sym, on="Patient_Num", how="outer") \
                   .merge(Ms_sym, on="Patient_Num", how="outer")

In [5]:
# 6. Save
data_abs.to_csv("FINAL_DELTA_ABS_CONCATENATED.csv", index=False)
data_rel.to_csv("FINAL_DELTA_REL_CONCATENATED.csv", index=False)
data_sym.to_csv("FINAL_DELTA_SYM_CONCATENATED.csv", index=False)
 
print(" Saved 3 CSVs ")
print("abs shape:", data_abs.shape)
print("rel shape:", data_rel.shape)
print("sym shape:", data_sym.shape)
 
# 7. check to see if each output is diffrent
num_abs = data_abs.select_dtypes(include=np.number).drop(columns=["Patient_Num"], errors="ignore")
num_rel = data_rel.select_dtypes(include=np.number).drop(columns=["Patient_Num"], errors="ignore")
num_sym = data_sym.select_dtypes(include=np.number).drop(columns=["Patient_Num"], errors="ignore")
 
# align on shared columns only 
sample_col_abs = [c for c in num_abs.columns if "firstorder_Mean" in c][0]
sample_col_rel = sample_col_abs.replace("_abs_", "_rel_")
sample_col_sym = sample_col_abs.replace("_abs_", "_sym_")
print(data_abs[["Patient_Num", sample_col_abs]].head())
print(data_rel[["Patient_Num", sample_col_rel]].head())
print(data_sym[["Patient_Num", sample_col_sym]].head())

 Saved 3 CSVs 
abs shape: (130, 4228)
rel shape: (130, 4228)
sym shape: (130, 4228)
   Patient_Num  main_abs_original_firstorder_MeanAbsoluteDeviation
0          1.0                                        -182.698649 
1          2.0                                        -136.519804 
2          3.0                                           5.739533 
3          4.0                                         -58.418387 
4          5.0                                         -16.580499 
   Patient_Num  main_rel_original_firstorder_MeanAbsoluteDeviation
0          1.0                                          -0.849528 
1          2.0                                          -0.782429 
2          3.0                                           0.221899 
3          4.0                                          -0.555572 
4          5.0                                          -0.432945 
   Patient_Num  main_sym_original_firstorder_MeanAbsoluteDeviation
0          1.0                               